# 3. Tool Guardrails: 권한 기반 실행

이 노트북은 [`Week04 - AI 에이전트 예제`](../../../week04_타당성검토_에이전트기획/day04_4일차/AI%20에이전트%20예제/)에서 배운 **`@tool` + `create_agent` 패턴**을 그대로 재사용해서, 왜 Tool마다 "누가 쓸 수 있는지"를 코드 레벨에서 강제해야 하는지 확인합니다.

- **A 코드**: 위험도가 다른 여러 Tool을 전부 한 Agent에 묶어서 실행 → 실행 결과에서 보안 문제 확인
- **B 코드**: 같은 Tool 함수들에 권한 메타데이터만 추가하고, 역할(role)에 따라 Agent에 묶이는 Tool 목록 자체를 다르게 함 → 같은 요청이 더 이상 통하지 않는 것을 확인

## 왜 Tool마다 권한이 필요한가요?

Tool은 그냥 함수가 아니라 **"진짜 권한을 가진 스위치"**입니다. 검색 Tool은 눌러도 안전하지만, "직원 레코드 삭제" Tool은 잘못 누르면 실제 데이터가 사라집니다. 그런데 `create_agent(tools=[...])`에 Tool을 등록하는 순간, **Agent를 쓰는 사람이 누구든 그 Tool을 요청 한 번으로 실행시킬 수 있습니다.** 신입사원이 쓰는 챗봇이든 관리자가 쓰는 챗봇이든 코드가 똑같다면, 신입사원도 삭제 버튼을 누를 수 있는 셈입니다.

### Tool 위험도 등급 (예시)

| 등급 | 예시 | 최소 필요 권한 |
| --- | --- | --- |
| 안전 (조회) | 직원 검색 | 전체(일반 사원) |
| 조회 (민감) | 급여 리포트 조회 | analyst 이상 |
| 발송 (외부 영향) | 공지 이메일 발송 | manager 이상 |
| 삭제 (파괴적) | 직원 레코드 삭제 | admin 전용 |

**해결책은 System Prompt에 "일반 사원은 삭제하지 마세요"라고 적는 것이 아닙니다.** LLM은 프롬프트의 지시를 "참고"할 뿐 강제로 지킬 수 없기 때문입니다. 진짜 해결책은 **애초에 권한이 없는 사용자의 Agent에는 그 Tool 자체를 등록하지 않는 것**입니다 — Tool이 없으면 아무리 그럴듯하게 요청해도 호출할 방법이 없습니다.

## 실습 준비: Tool과 가짜 사내 DB 정의

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
)

In [3]:
# 실습용 가짜 사내 인사 DB
COMPANY_DB = {
    "E1001": {"name": "김철수", "department": "영업"},
    "E1002": {"name": "이영희", "department": "인사"},
    "E1003": {"name": "박민수", "department": "개발"},
}
EMAIL_LOG = []


def reset_demo_state():
    """실습을 반복할 수 있도록 DB/로그를 초기 상태로 되돌립니다."""
    COMPANY_DB.clear()
    COMPANY_DB.update({
        "E1001": {"name": "김철수", "department": "영업"},
        "E1002": {"name": "이영희", "department": "인사"},
        "E1003": {"name": "박민수", "department": "개발"},
    })
    EMAIL_LOG.clear()

In [4]:
from langchain.tools import tool


@tool
def search_employee(name: str) -> str:
    """[안전/조회] 이름으로 직원을 검색합니다."""
    for eid, info in COMPANY_DB.items():
        if info["name"] == name:
            return f"{eid}: {info}"
    return "일치하는 직원을 찾을 수 없습니다."


@tool
def get_salary_report(department: str) -> str:
    """[민감 조회] 부서별 급여 통계를 조회합니다."""
    return f"'{department}' 부서 평균 급여: 4,800만원 (실습용 더미 데이터)"


@tool
def send_notice_email(to: str, message: str) -> str:
    """[발송] 직원에게 공지 이메일을 발송합니다."""
    EMAIL_LOG.append({"to": to, "message": message})
    return f"{to}님에게 이메일을 발송했습니다."


@tool
def delete_employee_record(employee_id: str) -> str:
    """[삭제/파괴적] 직원 레코드를 영구 삭제합니다."""
    if employee_id in COMPANY_DB:
        del COMPANY_DB[employee_id]
        return f"{employee_id} 레코드를 영구 삭제했습니다."
    return f"{employee_id}를 찾을 수 없습니다."


ALL_TOOLS = [search_employee, get_salary_report, send_notice_email, delete_employee_record]

## [A] 가드레일 없는 버전

Tool 4개를 전부 하나의 Agent에 등록합니다. **역할(role) 개념이 아예 없습니다** — 이 Agent를 신입사원이 쓰든 관리자가 쓰든 코드는 완전히 동일합니다.

In [5]:
from langchain.agents import create_agent

SYSTEM_PROMPT = "당신은 사내 인사 시스템 어시스턴트입니다. 사용자의 요청을 주어진 도구를 사용해 처리하세요."

agent = create_agent(
    model=llm,
    tools=ALL_TOOLS,
    system_prompt=SYSTEM_PROMPT,
)

### [A] 실행: 갓 입사한 일반 사원이 같은 Agent를 사용

"현우"는 오늘 막 입사한 일반 사원입니다. 사내 챗봇(방금 만든 `agent`)에게 별생각 없이 아래처럼 요청합니다.

> LLM은 삭제처럼 되돌릴 수 없는 작업 앞에서 한 번 더 "정말 삭제할까요?"라고 되묻는 습관이 있을 수 있습니다. 이건 **모델이 친절하게 재확인하는 대화 매너**일 뿐, "이 사람이 삭제할 권한이 있는지"를 검사하는 것이 아닙니다 — 누구든 그냥 "네"라고 한 번 더 답하면 그대로 진행됩니다. 그래서 아래 실습은 자연스러운 대화 2턴으로 구성합니다.

In [6]:
from langchain_core.messages import HumanMessage


def request_deletion(target_agent):
    """삭제를 요청하고, 모델이 재확인을 물으면 같은 사용자가 한 번 더 "네"라고 답해주는 헬퍼.
    (모델이 바로 처리하면 2턴은 자동으로 생략됩니다 — 모델 응답 스타일 편차를 흡수하기 위한 코드일 뿐,
    권한 검사와는 무관합니다)"""
    turn1 = target_agent.invoke({"messages": [HumanMessage(content="E1002 직원 레코드를 삭제해줘.")]})
    print("[Agent, 1턴]", turn1["messages"][-1].content)
    if "E1002" not in COMPANY_DB:
        return
    turn2 = target_agent.invoke({"messages": turn1["messages"] + [HumanMessage(content="네, 삭제해 주세요.")]})
    print("[Agent, 2턴]", turn2["messages"][-1].content)

In [7]:
reset_demo_state()
print("삭제 전 COMPANY_DB:", COMPANY_DB)

request_deletion(agent)

print("삭제 후 COMPANY_DB:", COMPANY_DB)

삭제 전 COMPANY_DB: {'E1001': {'name': '김철수', 'department': '영업'}, 'E1002': {'name': '이영희', 'department': '인사'}, 'E1003': {'name': '박민수', 'department': '개발'}}
[Agent, 1턴] 직원 레코드를 영구 삭제하는 작업은 되돌릴 수 없는 중요한 작업입니다.  
E1002 직원 레코드를 정말 삭제하시겠습니까? (예/아니오)
[Agent, 2턴] E1002 직원 레코드가 영구적으로 삭제되었습니다. 필요하신 다른 도움이 있으면 알려 주세요.
삭제 후 COMPANY_DB: {'E1001': {'name': '김철수', 'department': '영업'}, 'E1003': {'name': '박민수', 'department': '개발'}}


**문제 확인**: `COMPANY_DB`에서 `E1002`가 실제로 사라졌다면, **일반 사원 권한으로 관리자만 할 수 있어야 할 "레코드 영구 삭제"가 실행된 것**입니다. Agent가 한 번 되물은 것은 안전장치처럼 보이지만, 실제로는 "정말요?"에 "네"라고만 답하면 그만인 **대화 매너**였을 뿐입니다. Agent 입장에서는 그냥 "사용자가 요청했고, 그 요청을 처리할 수 있는 Tool이 있어서" 실행했을 뿐입니다 — 요청한 사람이 그 작업을 할 **권한이 있는지는 애초에 검사한 적이 없기 때문**입니다.

## [B] 권한 기반 Tool 필터링 추가

**Tool 함수(`search_employee`, `delete_employee_record` 등) 자체는 한 글자도 바꾸지 않습니다.** 각 Tool에 필요한 최소 역할을 매핑해두고, 역할별로 Agent에 등록되는 Tool 목록 자체를 다르게 구성합니다.

In [8]:
ROLE_LEVELS = {"general": 0, "analyst": 1, "manager": 2, "admin": 3}

# Tool 이름 -> 최소 필요 역할
TOOL_MIN_ROLE = {
    "search_employee": "general",
    "get_salary_report": "analyst",
    "send_notice_email": "manager",
    "delete_employee_record": "admin",
}


def get_tools_for_role(role: str):
    """역할 등급 이상이 필요한 Tool만 걸러서 반환합니다."""
    role_level = ROLE_LEVELS[role]
    return [
        t for t in ALL_TOOLS
        if ROLE_LEVELS[TOOL_MIN_ROLE[t.name]] <= role_level
    ]


def build_agent_for_role(role: str):
    """역할에 맞는 Tool만 묶어서 Agent를 새로 만듭니다."""
    tools = get_tools_for_role(role)
    prompt = (
        f"{SYSTEM_PROMPT}\n"
        f"[권한 안내] 현재 사용자의 역할은 '{role}'이며, 당신에게는 이 역할에 허용된 도구만 주어져 있습니다."
    )
    return create_agent(model=llm, tools=tools, system_prompt=prompt)

In [9]:
for role in ROLE_LEVELS:
    tool_names = [t.name for t in get_tools_for_role(role)]
    print(f"{role:8s} -> {tool_names}")

general  -> ['search_employee']
analyst  -> ['search_employee', 'get_salary_report']
manager  -> ['search_employee', 'get_salary_report', 'send_notice_email']
admin    -> ['search_employee', 'get_salary_report', 'send_notice_email', 'delete_employee_record']


### [B-1] 실행: 동일한 일반 사원, 동일한 요청 → 이번엔 삭제가 불가능

`general` 역할의 Agent에는 애초에 `delete_employee_record` Tool이 등록되어 있지 않습니다.

In [10]:
reset_demo_state()
print("삭제 전 COMPANY_DB:", COMPANY_DB)

general_agent = build_agent_for_role("general")
request_deletion(general_agent)

print("삭제 후 COMPANY_DB:", COMPANY_DB)

삭제 전 COMPANY_DB: {'E1001': {'name': '김철수', 'department': '영업'}, 'E1002': {'name': '이영희', 'department': '인사'}, 'E1003': {'name': '박민수', 'department': '개발'}}
[Agent, 1턴] 죄송합니다만, 현재 저는 직원 레코드를 삭제하는 권한이 없습니다. 해당 작업은 관리자 권한이 있는 분께 요청해 주시기 바랍니다. 필요하시면 해당 직원의 정보를 조회해 드릴 수 있습니다. 알려주시면 도와드리겠습니다.
[Agent, 2턴] 죄송합니다만, 저는 직원 레코드를 삭제하는 기능에 접근할 수 없습니다. 해당 작업은 관리자 권한이 있는 분께 요청해 주시기 바랍니다. 필요하시면 E1002 직원의 정보를 조회해 드릴 수 있으니 알려주세요.
삭제 후 COMPANY_DB: {'E1001': {'name': '김철수', 'department': '영업'}, 'E1002': {'name': '이영희', 'department': '인사'}, 'E1003': {'name': '박민수', 'department': '개발'}}


**문제 확인**: `COMPANY_DB`가 그대로 남아 있어야 합니다. `general_agent`는 `delete_employee_record`라는 Tool의 존재 자체를 모르기 때문에, 아무리 "삭제해줘"라고 요청해도 **호출할 방법이 없습니다.** System Prompt로 타이르는 것과 달리, 이건 우회할 수 있는 "설득"이 아니라 "물리적으로 불가능한" 구조적 제약입니다.

### [B-2] 실행: 정당한 관리자는 동일한 작업을 정상적으로 수행

In [11]:
admin_agent = build_agent_for_role("admin")
request_deletion(admin_agent)

print("삭제 후 COMPANY_DB:", COMPANY_DB)

[Agent, 1턴] E1002 직원 레코드가 영구적으로 삭제되었습니다. 필요하신 다른 작업이 있으면 알려주세요.
삭제 후 COMPANY_DB: {'E1001': {'name': '김철수', 'department': '영업'}, 'E1003': {'name': '박민수', 'department': '개발'}}


**확인**: `admin` 역할의 Agent에는 `delete_employee_record`가 포함되어 있으므로 정상적으로 삭제됩니다. 즉 권한 기반 Tool 필터링은 기능을 막는 것이 아니라, **적절한 권한을 가진 사람에게만 열어주는 것**입니다.

## 정리

| | A (가드레일 없음) | B (가드레일 추가) |
| --- | --- | --- |
| Tool 함수 코드 | 강의 패턴 그대로 | 완전히 동일 (수정 없음) |
| Agent 구성 | 역할 구분 없이 모든 Tool을 하나의 Agent에 등록 | 역할별로 `get_tools_for_role()`이 필터링한 Tool만 등록 |
| 일반 사원이 삭제 요청 | 실제로 삭제됨 | Tool이 없어서 애초에 실행 불가 |
| 관리자가 삭제 요청 | 실제로 삭제됨 | 정상적으로 삭제됨(기능은 유지) |

**기억할 점**: System Prompt로 "이 역할은 이거 하면 안 돼"라고 지시하는 것은 **참고용 가이드라인일 뿐 강제력이 없습니다.** 진짜 권한 통제는 애초에 "그 사용자의 Agent 인스턴스에 그 Tool을 바인딩하지 않는" 코드 레벨의 구조적 제약으로 구현해야 합니다. 다음 노트북(4. SQL/Python Tool 최소 권한)에서는 Tool 하나의 **내부** 로직에서 최소 권한을 강제하는 방법을 다룹니다.